In [0]:
proc_df = spark.table("medical_project.silver.procedures")
enc_df = spark.table("medical_project.silver.encounters")
display(proc_df)

In [0]:
fact_proc = proc_df.join(
    enc_df,
    proc_df.encounter_id == enc_df.id,
    "left"
)

display(fact_proc)

In [0]:
# Select Required Columns
from pyspark.sql.functions import col, to_date

# Re-do the join with proper column selection to avoid ambiguity
fact_proc_clean = proc_df.alias("p").join(
    enc_df.alias("e"),
    col("p.encounter_id") == col("e.id"),
    "left"
).select(
    col("p.encounter_id"),
    col("p.patient").alias("patient_id"),
    col("p.code").alias("procedure_code"),
    col("p.description").alias("procedure_description"),
    col("p.base_cost"),
    to_date(col("p.start")).alias("procedure_date")
)

# Handle Nulls
fact_proc = fact_proc_clean.fillna({
    "base_cost": 0
})

display(fact_proc)

In [0]:
# Save Table
fact_proc.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.fact_procedures")